# State Objective Checkpoint Selection v1

"
        "This notebook converts state-objective alignment scores into a checkpoint/candidate selection objective. It is a post-screen for retained checkpoints and candidate daily paths, designed to avoid selecting by validation score alone when the model is overfit.

"
        "Governance:
"
        "- canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`
"
        "- direct comparison baseline: `state_objective_alignment_screen_v1`
"
        "- role: `diagnose`
"
        "- changed layer: `selection`
"
        "- accepted trade-off: no runtime promotion unless both state-target coverage and 30pp/15% window gates pass

"
        "External basis: Qlib artifact-first workflow, FinRL reward/action separation, and Probability of Backtest Overfitting OOS/walk-forward discipline.
"
        

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / 'src' / 'soxl_quant').exists():
        REPO_ROOT = parent
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.state_objective_training_targets import (
    score_state_objective_checkpoint_candidates,
    summarize_state_objective_checkpoint_selection,
)

EXP = 'state_objective_checkpoint_selection_v1'
OUT_DIR = REPO_ROOT / 'runtime/model_selection/card' / EXP
ALIGN_DIR = REPO_ROOT / 'runtime/model_selection/card/state_objective_alignment_screen_v1'
TARGET_DIR = REPO_ROOT / 'runtime/model_selection/card/state_objective_training_target_v1'
SCREEN_DIR = REPO_ROOT / 'runtime/model_selection/card/state_conditioned_objective_screen_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

alignment_selection = json.loads((ALIGN_DIR / 'selection.json').read_text())
target_selection = json.loads((TARGET_DIR / 'selection.json').read_text())
screen_selection = json.loads((SCREEN_DIR / 'selection.json').read_text())
alignment_scores = pd.read_csv(ALIGN_DIR / 'alignment_scores.csv')
alignment_scores.head()


,candidate,target_rows,matched_rows,missing_target_ratio,coverage_gate,position_tracking_mae,rebound_position_mae,risk_cap_violation,window_gate_loss,max_relative_gap,max_drawdown_gap,window_gate,alignment_loss
0,state_conditioned_rebound_risk_router_v1,2322,2322,0.0,True,0.000000,0.151144,0.015691,134.348893,77.569598,2.939129,False,1.708542
1,recovery_reentry_conditional_vol_target_v1,2322,2322,0.0,True,0.253706,0.208537,0.270763,356.920108,0.000000,15.923275,False,5.323034
2,recovery_reentry_cppi_floor_v1,2322,2322,0.0,True,0.265752,0.261291,0.279327,404.543049,0.000000,17.120180,False,5.951073
3,current_coverage_residual_guard_refresh_v1,2322,2322,0.0,True,0.201421,0.207981,0.192123,483.891119,155.130239,19.266345,False,6.224786
4,recovery_reentry_equity_drawdown_guard_v1,2322,2322,0.0,True,0.194548,0.178789,0.189878,565.721828,45.413925,19.365251,False,6.968854


In [2]:
checkpoint_scores = score_state_objective_checkpoint_candidates(alignment_scores)
summary = summarize_state_objective_checkpoint_selection(checkpoint_scores)

scores_path = OUT_DIR / 'checkpoint_objective_scores.csv'
summary_path = OUT_DIR / 'summary.json'
recipe_path = OUT_DIR / 'training_objective_recipe.json'
checkpoint_scores.to_csv(scores_path, index=False)
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

recipe = {
    'objective': 'state_objective_checkpoint_or_bc',
    'selected_teacher_candidate': summary['selected_candidate'],
    'promotion_allowed': summary['decision'] == 'checkpoint_objective_promotion_review',
    'loss_terms': summary['required_terms'],
    'selection_score': {
        'primary': 'minimize checkpoint_objective_loss',
        'alignment_loss_weight': 1.0,
        'window_gate_loss_weight': 0.01,
        'missing_target_penalty': 10.0,
        'rebound_mae_weight': 1.0,
        'risk_cap_violation_weight': 2.0,
        'legacy_score_reward': 0.0,
    },
    'gates': {
        'coverage_gate_required': True,
        'window_gate_required_for_promotion': True,
        'target_relative_return_min_pp': target_selection['target']['relative_return_min_pp'],
        'target_max_dd_max_pct': target_selection['target']['max_dd_max_pct'],
    },
    'next_training_use': [
        'use training_targets.csv for BC warm-start target labels or checkpoint scoring',
        'penalize risk-cap violations on downtrend target rows',
        'penalize rebound target underparticipation on rebound cluster rows',
        'retain checkpoints by checkpoint_objective_loss, not validation score alone',
    ],
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding='utf-8')
summary


{'decision': 'checkpoint_objective_no_promotion',
 'selected_candidate': 'state_conditioned_rebound_risk_router_v1',
 'selected_checkpoint_objective_loss': 3.2345572516904033,
 'selected_checkpoint_selection_score': -3.2345572516904033,
 'selected_window_gate': False,
 'selected_coverage_gate': True,
 'selected_alignment_loss': 1.70854185633442,
 'selected_window_gate_loss': 134.34889343775464,
 'selected_missing_target_ratio': 0.0,
 'next_mechanism': 'train_state_objective_checkpoint_or_bc',
 'required_terms': ['checkpoint_objective_alignment_loss',
  'checkpoint_objective_walkforward_gate_loss',
  'checkpoint_objective_rebound_mae',
  'checkpoint_objective_risk_cap_violation']}

In [3]:
display(Markdown('## Checkpoint Objective Ranking'))
display(checkpoint_scores)

display(Markdown('## Training Objective Recipe'))
display(pd.DataFrame([recipe['selection_score']]))


## Checkpoint Objective Ranking

,candidate,target_rows,matched_rows,missing_target_ratio,coverage_gate,position_tracking_mae,rebound_position_mae,risk_cap_violation,window_gate_loss,max_relative_gap,max_drawdown_gap,window_gate,alignment_loss,score,checkpoint_objective_loss,checkpoint_selection_score
0,state_conditioned_rebound_risk_router_v1,2322,2322,0.0,True,0.000000,0.151144,0.015691,134.348893,77.569598,2.939129,False,1.708542,0.0,3.234557,-3.234557
1,recovery_reentry_conditional_vol_target_v1,2322,2322,0.0,True,0.253706,0.208537,0.270763,356.920108,0.000000,15.923275,False,5.323034,0.0,9.642299,-9.642299
2,recovery_reentry_cppi_floor_v1,2322,2322,0.0,True,0.265752,0.261291,0.279327,404.543049,0.000000,17.120180,False,5.951073,0.0,10.816449,-10.816449
3,current_coverage_residual_guard_refresh_v1,2322,2322,0.0,True,0.201421,0.207981,0.192123,483.891119,155.130239,19.266345,False,6.224786,0.0,11.655924,-11.655924
4,recovery_reentry_equity_drawdown_guard_v1,2322,2322,0.0,True,0.194548,0.178789,0.189878,565.721828,45.413925,19.365251,False,6.968854,0.0,13.184616,-13.184616
5,recovery_reentry_anchor_confirm_v1,2322,2322,0.0,True,0.273907,0.204303,0.310833,608.266964,0.000000,26.587207,False,8.008514,0.0,14.917152,-14.917152
6,current_coverage_rebound_downtrend_router_v1,2322,2322,0.0,True,0.188386,0.207739,0.173488,703.795741,227.306769,20.986375,False,8.335772,0.0,15.928443,-15.928443
7,state_conditioned_actor_selector_v1,2322,2322,0.0,True,0.268944,0.203634,0.314367,970.446343,18.644654,34.774907,False,11.638144,0.0,22.174976,-22.174976


## Training Objective Recipe

,primary,alignment_loss_weight,window_gate_loss_weight,missing_target_penalty,rebound_mae_weight,risk_cap_violation_weight,legacy_score_reward
0,minimize checkpoint_objective_loss,1.0,0.01,10.0,1.0,2.0,0.0


In [4]:
selection = {
    'experiment': EXP,
    'decision': summary['decision'],
    'promotion': 'blocked_no_window_gate' if summary['decision'] != 'checkpoint_objective_promotion_review' else 'review_required',
    'role': 'diagnose',
    'changed_layer': ['selection'],
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'state_objective_alignment_screen_v1',
    'teacher_candidate': target_selection['teacher_candidate'],
    'selected_candidate': summary['selected_candidate'],
    'target': target_selection['target'],
    'summary': summary,
    'artifacts': {
        'notebook': str((OUT_DIR / f'{EXP}.ipynb').relative_to(REPO_ROOT)),
        'checkpoint_objective_scores': str(scores_path.relative_to(REPO_ROOT)),
        'training_objective_recipe': str(recipe_path.relative_to(REPO_ROOT)),
        'summary': str(summary_path.relative_to(REPO_ROOT)),
        'selection': str((OUT_DIR / 'selection.json').relative_to(REPO_ROOT)),
    },
    'external_references': screen_selection.get('external_references', []),
    'conclusion': [
        'The candidate/checkpoint selector now consumes state-objective alignment loss instead of validation score alone.',
        'No candidate is promotion-ready because the selected row still fails the walk-forward/window gate.',
        'The next training branch should retain/checkpoint by checkpoint_objective_loss and then rerun walk-forward validation.',
    ],
}
(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')

display(Markdown(
    f"""## Conclusion\n\n"
    f"- decision: `{selection['decision']}`\n"
    f"- selected candidate/checkpoint teacher: `{selection['selected_candidate']}`\n"
    f"- checkpoint objective loss: `{summary['selected_checkpoint_objective_loss']:.4f}`\n"
    f"- window gate: `{summary['selected_window_gate']}`; coverage gate: `{summary['selected_coverage_gate']}`\n"
    f"- next mechanism: `{summary['next_mechanism']}`\n\n"
    "This is a training/checkpoint-selection objective, not runtime promotion. The next PPO/BC branch should use this loss for retained checkpoint selection and rerun walk-forward validation."""
))
selection


## Conclusion

"
    f"- decision: `checkpoint_objective_no_promotion`
"
    f"- selected candidate/checkpoint teacher: `state_conditioned_rebound_risk_router_v1`
"
    f"- checkpoint objective loss: `3.2346`
"
    f"- window gate: `False`; coverage gate: `True`
"
    f"- next mechanism: `train_state_objective_checkpoint_or_bc`

"
    "This is a training/checkpoint-selection objective, not runtime promotion. The next PPO/BC branch should use this loss for retained checkpoint selection and rerun walk-forward validation.

{'experiment': 'state_objective_checkpoint_selection_v1',
 'decision': 'checkpoint_objective_no_promotion',
 'promotion': 'blocked_no_window_gate',
 'role': 'diagnose',
 'changed_layer': ['selection'],
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'state_objective_alignment_screen_v1',
 'teacher_candidate': 'state_conditioned_rebound_risk_router_v1',
 'selected_candidate': 'state_conditioned_rebound_risk_router_v1',
 'target': {'relative_return_min_pp': 30.0, 'max_dd_max_pct': 15.0},
 'summary': {'decision': 'checkpoint_objective_no_promotion',
  'selected_candidate': 'state_conditioned_rebound_risk_router_v1',
  'selected_checkpoint_objective_loss': 3.2345572516904033,
  'selected_checkpoint_selection_score': -3.2345572516904033,
  'selected_window_gate': False,
  'selected_coverage_gate': True,
  'selected_alignment_loss': 1.70854185633442,
  'selected_window_gate_loss': 134.34889343775464,
  'selected_missing_target_ratio': 0.0,
  'ne